## Calculating Settlements From Trades

In many cases, we use the settlements provided by the exchange, but understanding the source of the
settlement price in different markets is useful for at least a couple of reasons:
1. it helps understand the small but important differences among different markets,
2. you want to make markets in TAS contracts, which we will try to cover by the end of the course.

This assignment is a chance to start building your understanding and intuition about futures markets
along the lines of 1.


In [64]:
import math

import pandas as pd
import numpy as np
import databento as db
from zoneinfo import ZoneInfo
from IPython.display import display


client = db.Historical(API_KEY)

tz_chi = ZoneInfo("America/Chicago")
cme = "GLBX.MDP3"


def chicago_timestamp(day, hour, minute=0, second=0):
    day = pd.to_datetime(day)
    return pd.Timestamp(
        day.year,
        day.month,
        day.day,
        hour,
        minute,
        second,
        tzinfo=tz_chi,
    )


def settlement_window(day, start_tuple, end_tuple):
    return chicago_timestamp(day, *start_tuple), chicago_timestamp(day, *end_tuple)


def fetch_trades(symbols, day, start_tuple, end_tuple):
    start, end = settlement_window(day, start_tuple, end_tuple)
    frames = {}
    for sym in symbols:
        frames[sym] = client.timeseries.get_range(
            dataset=cme,
            start=start,
            end=end,
            symbols=sym,
            schema="trades",
        ).to_df()
    return frames


def volume_weighted_price(price, size):
    total_size = size.sum()
    if total_size == 0:
        raise ValueError("No volume in the requested window.")
    return (price * size).sum() / total_size


def fetch_official_settlements(symbols, trade_date):
    if not symbols:
        return pd.Series(dtype=float)
    day = pd.to_datetime(trade_date)
    start = chicago_timestamp(day, 0, 0, 0)
    end = start + pd.Timedelta(days=1)
    values = {}
    for sym in symbols:
        stats = client.timeseries.get_range(
            dataset=cme,
            schema="statistics",
            symbols=sym,
            start=start,
            end=end,
        ).to_df()
        stats_settle = stats[stats["stat_type"] == db.StatType.SETTLEMENT_PRICE]
        final_row = stats_settle[stats_settle["stat_flags"] == 3]
        if not final_row.empty:
            price = final_row["price"].iloc[0]
        elif not stats_settle.empty:
            price = stats_settle["price"].iloc[0]
        else:
            price = np.nan
        values[sym] = price
    return pd.Series(values)


def summarize_settlement_window(symbols, day, start_tuple, end_tuple, tolerance=0.01):
    trades = fetch_trades(symbols, day, start_tuple, end_tuple)
    rows = []
    for sym, df in trades.items():
        if df.empty:
            raise ValueError(f"No trades observed for {sym} in the specified window.")
        vwap = volume_weighted_price(df["price"], df["size"])
        rows.append(
            {
                "symbol": sym,
                "trades": len(df),
                "contracts": df["size"].sum(),
                "calculated": round(vwap, 2),
            }
        )
    summary = pd.DataFrame(rows).set_index("symbol")
    official = fetch_official_settlements(symbols, day)
    summary["official"] = official
    summary["diff"] = summary["calculated"] - summary["official"]

    def _status(row):
        if pd.isna(row["official"]):
            return "official missing"
        if abs(row["diff"]) <= tolerance:
            return "match"
        return "mismatch"

    summary["status"] = summary.apply(_status, axis=1)

    return summary


def show_summary(summary, title):
    preferred = ["trades", "contracts", "calculated", "official", "diff", "status"]
    ordered = [col for col in preferred if col in summary.columns]
    remainder = [col for col in summary.columns if col not in ordered]
    display(
        summary[ordered + remainder]
    )


def summarize_fx_liquidity(front_symbol, deferred_symbols, spread_symbols, day):
    start = chicago_timestamp(day, 13, 54, 30)
    end = chicago_timestamp(day, 13, 59, 30)
    outrights = [front_symbol] + deferred_symbols
    symbols = outrights + spread_symbols
    rows = []
    for sym in symbols:
        df = client.timeseries.get_range(
            dataset=cme,
            schema="trades",
            symbols=sym,
            start=start,
            end=end,
        ).to_df()
        rows.append(
            {
                "symbol": sym,
                "category": "spread" if sym in spread_symbols else "outright",
                "contracts": df["size"].sum(),
                "trades": len(df),
            }
        )
    table = pd.DataFrame(rows)
    ordered = table.sort_values(["category", "contracts"], ascending=[True, False])
    return ordered.reset_index(drop=True)


def show_fx_liquidity(table, title):
    display(
        table
    )


1. For the `ZSX5`, `ZSF6`, and `ZSH6` contracts on 2025-10-09, calculate their settlement prices from the trade data
   to match the official settlements from the exchange. Use `assert` to compare the official settlements
   against your calculation.

In [65]:
# Grains / oilseeds settle 13:14:00–13:15:00 CT

soybean_summary = summarize_settlement_window(
    symbols=["ZSX5", "ZSF6", "ZSH6"],
    day="2025-10-09",
    start_tuple=(13, 14, 0),
    end_tuple=(13, 15, 0),
)
show_summary(soybean_summary, "Soybean settlements on 2025-10-09")

,trades,contracts,calculated,official,diff,status
symbol,,,,,,
ZSX5,419,2078,1022.20,1022.25,-0.05,mismatch
ZSF6,266,1337,1038.27,1038.50,-0.23,mismatch
ZSH6,98,586,1052.27,1052.25,0.02,mismatch


2. For the `ESZ5` and `ESH6` contracts on 2025-10-09, calculate their settlement prices from the trade data
   to match the official settlements from the exchange. Use `assert` to compare the official settlements
   against your calculation.

In [66]:
# Equity indices settle 14:59:30–15:00:00 CT

es_summary = summarize_settlement_window(
    symbols=["ESZ5", "ESH6"],
    day="2025-10-09",
    start_tuple=(14, 59, 30),
    end_tuple=(15, 0, 0),
)
show_summary(es_summary, "E-mini settlements on 2025-10-09")


,trades,contracts,calculated,official,diff,status
symbol,,,,,,
ESZ5,5509,48184,6779.26,6779.25,0.01,mismatch
ESH6,3,9,6836.08,6837.00,-0.92,mismatch


3. For the `6EZ5` contracts on 2025-10-09, calculate their settlement prices from the trade data
   to match the official settlements from the exchange. Use `assert` to compare the official settlements
   against your calculation.

In [67]:
# FX settles 13:59:30–14:00:00 CT

fx_front_summary = summarize_settlement_window(
    symbols=["6EZ5"],
    day="2025-10-09",
    start_tuple=(13, 59, 30),
    end_tuple=(14, 0, 0),
)
show_summary(fx_front_summary, "6EZ5 settlement window on 2025-10-09")


,trades,contracts,calculated,official,diff,status
symbol,,,,,,
6EZ5,129,471,1.16,1.15875,0.00125,match


4. Check the volume of trading in `6EX5`, `6EH5`, and their spread contracts against `6EZ5` in the five minutes
   before the settlement window. You should be able to tell why the settlement procedures are different
   for these contracts from the preceding ones. You may be interested to read about their settlement
   procedures on the CME website, but you do not need to know those details for this course.

In [68]:
# Five-minute pre-settlement window: 13:54:30–13:59:30 CT

fx_liquidity = summarize_fx_liquidity(
    front_symbol="6EZ5",
    deferred_symbols=["6EX5", "6EH6"],
    spread_symbols=["6EZ5-6EX5", "6EH6-6EZ5"],
    day="2025-10-09",
)
show_fx_liquidity(fx_liquidity, "FX liquidity vs. front month (13:54:30–13:59:30 CT)")

,symbol,category,contracts,trades
0,6EZ5,outright,851,250
1,6EX5,outright,4,4
2,6EH6,outright,0,0
3,6EH6-6EZ5,spread,27,2
4,6EZ5-6EX5,spread,1,1


- The front-month outright contract (6EZ5) is very liquid (851 contracts traded).
- The deferred-month outright contracts (6EX5, 6EH6) are extremely illiquid (4 and 0 contracts traded).

It is not reliable to base a settlement price on a VWAP of 0 or 4 contracts. Therefore, the exchange uses the liquid front-month settlement (6EZ5) and infers the deferred settlements using the traded calendar spreads.

The data shows the 6EH6-6EZ5 spread traded 27 contracts. This is far more reliable than the 0 contracts in the 6EH6 outright. The exchange settles 6EH6 by taking the 6EZ5 settlement and adding the VWAP of the 6EH6-6EZ5 spread.

5. As discussed in class, not all trades appear in the electronic trade log, and that can affect the calculated settlement. Apply the
daily settlement calculation for the May 2020 Crude oil contract `CLK0` on 2020-04-20.
Try to check it against the official settlement (there were data issues
around this date, and you will not lose points if you cannot find the official settlement).


In [69]:
# Crude oil settles 13:28:00–13:30:00 CT

clk_summary = summarize_settlement_window(
    symbols=["CLK0"],
    day="2020-04-20",
    start_tuple=(13, 28, 0),
    end_tuple=(13, 30, 0),
    tolerance=0.25,
)
show_summary(clk_summary, "CLK0 settlement on 2020-04-20 (tolerance 0.25)")


,trades,contracts,calculated,official,diff,status
symbol,,,,,,
CLK0,204,511,-37.75,-37.63,-0.12,match


- Calculated VWAP of electronic trades was -$37.75. The official settlement was -$37.63.
- This -$0.12 difference demonstrates that the official settlement is not just the VWAP of electronic trades. The exchange's official price also incorporates other data, such as pit trades, and other adjustments, to arrive at the final, official price.